# Modelagem Temporal de Áudio com LSTM
**Módulo:** Deep Learning — Classificação de Gêneros Musicais (GTZAN)
**Responsável:** Caio Rosendo

---

## 1. Propósito e Objetivos
Este notebook explora a modelagem temporal de áudios musicais utilizando Redes Neurais Recorrentes (RNNs), especificamente uma arquitetura **Bi-LSTM** (Bidirectional Long Short-Term Memory). O objetivo principal é avaliar a capacidade de uma rede recorrente em extrair padrões sequenciais frame a frame a partir de coeficientes cepstrais (MFCCs).

Neste pipeline, utilizamos os dados pré-processados pela etapa de *Features & Baseline* (Araújo). As músicas do dataset GTZAN foram divididas em segmentos de 3 segundos, convertidas em sequências de MFCCs (shape `130 frames x 34 features`), e separadas em 3 *folds* respeitando o **filtro de artistas** (método Foleiss/Sturm) para evitar vazamento de dados (*data leakage*).

**Nossos objetivos específicos são:**
* Processar a dimensionalidade do tempo bruto do áudio.
* Implementar técnicas robustas de regularização para combater o forte sobreajuste (*overfitting*) comum no dataset GTZAN.
* Agregar as predições dos segmentos (Soft Majority Voting) para definir o gênero final da música.
* Comparar a eficiência temporal da LSTM contra o modelo de visão espacial (CNN do Borges) e os modelos tabulares clássicos (SVM/RF do Araújo).

## 2. Configuração do Ambiente e Extração de Features
As células abaixo preparam o ambiente de execução e resgatam as matrizes `.npy` geradas na etapa de baseline. O uso de matrizes NumPy pré-computadas acelera drasticamente o processo de treinamento, pois a extração dos MFCCs via `librosa` é computacionalmente custosa e não precisa ser refeita a cada época.

O código agrupa as partes compactadas para contornar os limites de tamanho do repositório, descompacta os arquivos forçando a sobrescrita (flag `-qo`) para automatizar a execução na nuvem, e limpa o armazenamento em disco.

In [1]:
# Célula 1: Setup de Ambiente
from google.colab import drive
import os

# 1. Monta o Google Drive
drive.mount('/content/drive')

# 2. Clona o repositório da equipe
!git clone https://github.com/caiorj2004/trabalho_deep_learning.git /content/trabalho_deep_learning

print("\nRepositório clonado com sucesso!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
fatal: destination path '/content/trabalho_deep_learning' already exists and is not an empty directory.

Repositório clonado com sucesso!


In [2]:
# Célula 2: Instalação e Dados
!pip install -q librosa torch torchvision torchaudio scikit-learn matplotlib seaborn tqdm pandas numpy

# Navega para a pasta onde o Araújo deixou os zips
%cd /content/trabalho_deep_learning/features_baseline

# Junta as duas partes do zip
!cat mfcc_seq.zip.001 mfcc_seq.zip.002 > mfcc_seq.zip

# Descompacta gerando a pasta mfcc_seq/ com os arquivos .npy
!unzip -qo mfcc_seq.zip

# Remove o zip reconstruído para poupar espaço em disco no Colab
!rm mfcc_seq.zip

# Retorna para o diretório raiz do Colab
%cd /content
print("\nArquivos .npy descompactados e prontos para uso!")

/content/trabalho_deep_learning/features_baseline
/content

Arquivos .npy descompactados e prontos para uso!


## 3. Dependências e Rastreamento de Arquivos
Aqui importamos o ecossistema PyTorch para a modelagem profunda, o `scikit-learn` para as métricas de avaliação e o `plotly` para a visualização iterativa de dados.

O código rastreia dinamicamente o arquivo `manifest.csv`. Esse manifesto é o coração do nosso rigor metodológico: ele dita a qual *fold* cada música pertence, garantindo que nenhum segmento do mesmo artista apareça simultaneamente nos conjuntos de treino e teste.

In [3]:
# Célula 3: Imports e Caminhos
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")

# Rastreia automaticamente o manifesto e os arquivos
manifest_matches = list(Path('/content/trabalho_deep_learning').rglob('manifest.csv'))
if not manifest_matches:
    raise FileNotFoundError("Não achei o manifest.csv! Você rodou a Célula 1?")
MANIFEST_PATH = manifest_matches[0]

npy_matches = list(Path('/content/trabalho_deep_learning').rglob('blues.00000.npy'))
if not npy_matches:
    raise FileNotFoundError("Não achei os arquivos .npy! Você rodou a Célula 2?")
MFCC_DIR = npy_matches[0].parent

OUTPUT_DIR = Path("/content/drive/MyDrive/gtzan/outputs_lstm")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"✅ Device: {DEVICE}")

GENRES = ['blues', 'classical', 'country', 'disco', 'hiphop',
          'jazz', 'metal', 'pop', 'reggae', 'rock']
GENRE2IDX = {g: i for i, g in enumerate(GENRES)}
IDX2GENRE = {i: g for g, i in GENRE2IDX.items()}

✅ Device: cuda


## 4. Modelagem e Estratégias Arquiteturais (A LSTM)
A classe `MusicLSTM` é o núcleo deste notebook. Processar áudio frame a frame é um desafio, pois transições em nível de milissegundos são ruidosas. Para extrair conhecimento útil e evitar que a rede "decore" o dataset, implementamos as seguintes escolhas arquiteturais:

1. **Bi-LSTM:** A rede avalia a música nas duas direções do tempo (frente e trás), permitindo que o contexto de um acorde futuro influencie a interpretação do frame atual.
2. **Mean Pooling (Média Temporal):** Em vez de utilizar apenas o *hidden state* do último frame (prática comum em NLP, mas falha em música, pois o gênero não se resume ao último milissegundo), nós tiramos a **média global de todos os 130 estados ocultos**. Isso força a rede a aprender a "assinatura geral" do trecho de 3 segundos.
3. **Data Augmentation 1D (Injeção de Ruído):** Adicionamos um leve ruído Gaussiano às features diretamente na etapa de treino (implementado no loop da célula seguinte). Isso dificulta a memorização exata dos valores acústicos, agindo como um poderoso regularizador.
4. **Regularização Agressiva:** Utilizamos `Dropout` alto (0.5 a 0.6) e normalização em camadas (`LayerNorm`) para evitar o colapso dos pesos e forçar a rede a generalizar padrões.

In [4]:
# Célula 4: Dataset e Modelo
class GTZANSequentialDataset(Dataset):
    def __init__(self, manifest_df, mfcc_dir, mode="train"):
        self.mfcc_dir = Path(mfcc_dir)
        self.mode = mode
        self.samples = []
        for _, r in manifest_df.iterrows():
            npy_path = self.mfcc_dir / f"{r['clip']}.npy"
            if npy_path.exists():
                self.samples.append({"clip": r["clip"], "genre": r["genre"], "path": npy_path})

    def __len__(self):
        return len(self.samples) * 10 if self.mode == "train" else len(self.samples)

    def __getitem__(self, idx):
        if self.mode == "train":
            clip_idx, seg_idx = idx // 10, idx % 10
            sample = self.samples[clip_idx]
            data = np.load(sample["path"]).astype(np.float32)
            return torch.tensor(data[seg_idx]), GENRE2IDX[sample["genre"]]
        else:
            sample = self.samples[idx]
            data = np.load(sample["path"]).astype(np.float32)
            return torch.tensor(data), GENRE2IDX[sample["genre"]]

class MusicLSTM(nn.Module):
    def __init__(self, input_size=34, hidden_size=64, num_layers=2, n_classes=10):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size, hidden_size=hidden_size,
            num_layers=num_layers, batch_first=True,
            bidirectional=True, dropout=0.5
        )
        lstm_out = hidden_size * 2
        self.classifier = nn.Sequential(
            nn.LayerNorm(lstm_out),
            nn.Dropout(0.5),
            nn.Linear(lstm_out, 32),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(32, n_classes)
        )

    def forward(self, x):
        # lstm_out shape: (batch, 130_frames, hidden_size*2)
        lstm_out, _ = self.lstm(x)

        # Mean Pooling
        # Em vez de pegar o último frame, tiramos a média de todo o eixo temporal (dim=1)
        h_mean = lstm_out.mean(dim=1)

        return self.classifier(h_mean)

def compute_stats(manifest_df, mfcc_dir):
    seqs = [np.load(Path(mfcc_dir) / f"{row['clip']}.npy").reshape(-1, 34)
            for _, row in manifest_df.iterrows() if (Path(mfcc_dir) / f"{row['clip']}.npy").exists()]
    X = np.concatenate(seqs, axis=0)
    return X.mean(axis=0), X.std(axis=0) + 1e-8

class NormalizedDataset(Dataset):
    def __init__(self, dataset, mean, std):
        self.dataset = dataset
        self.mean, self.std = torch.tensor(mean, dtype=torch.float32), torch.tensor(std, dtype=torch.float32)

    def __len__(self): return len(self.dataset)
    def __getitem__(self, idx):
        x, y = self.dataset[idx]
        return (x - self.mean) / self.std, y

## 5. Dinâmica de Treino e *Majority Voting*
Nesta etapa, definimos as funções centrais de fluxo de dados:

* **Função de Treino:** Responsável pela injeção do ruído (Data Augmentation), pelo cálculo da entropia cruzada (*CrossEntropyLoss*) e pela aplicação do *Gradient Clipping* (max_norm=1.0) para evitar explosão de gradientes nas retropropagações da LSTM.
* **Avaliação de Clipes (*evaluate_clips*):** Para prever a música inteira (30s), o modelo processa os 10 segmentos de 3 segundos individualmente. Em seguida, as probabilidades (*logits*) são somadas e a classe com maior pontuação acumulada vence. Esta técnica de **Soft Majority Voting** é muito mais estável do que classificar a música a partir de um único trecho aleatório.

In [5]:
# Célula 5: Treinamento e Avaliação
def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)

        # Adiciona um ruído leve (std=0.1) apenas durante o treino (Data Augmentation 1D)
        noise = torch.randn_like(x) * 0.1
        x_noisy = x + noise

        optimizer.zero_grad()
        logits = model(x_noisy)
        loss = criterion(logits, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item() * len(y)
        correct += (logits.argmax(1) == y).sum().item()
        total   += len(y)
    return total_loss / total, correct / total

@torch.no_grad()
def evaluate_clips(model, manifest_df, mfcc_dir, mean, std, device):
    model.eval()
    mean_t, std_t = torch.tensor(mean, dtype=torch.float32, device=device), torch.tensor(std, dtype=torch.float32, device=device)
    all_true, all_pred = [], []

    for _, row in manifest_df.iterrows():
        path = Path(mfcc_dir) / f"{row['clip']}.npy"
        if not path.exists(): continue

        segs = torch.tensor(np.load(path).astype(np.float32), device=device)
        segs = (segs - mean_t) / std_t

        logits = model(segs)
        clip_pred = logits.sum(0).argmax().item() # Majority Voting / Soft Voting

        all_true.append(GENRE2IDX[row["genre"]])
        all_pred.append(clip_pred)

    acc = accuracy_score(all_true, all_pred)
    f1  = f1_score(all_true, all_pred, average="macro", zero_division=0)
    return acc, f1, all_true, all_pred

## 6. Validação 3-Fold e Exportação de Resultados
Executamos o treinamento sob um rigoroso sistema de validação cruzada (3-Fold Cross-Validation). Para otimizar a convergência, utilizamos o otimizador **AdamW** com um `weight_decay` de 1e-3 (penalidade forte nos pesos, combatendo o overfitting) e o *scheduler* **Cosine Annealing** para ajustar a taxa de aprendizado gradativamente.

O sistema conta com *Early Stopping* de 15 épocas, garantindo que o modelo seja salvo no seu pico exato de generalização. Ao final de cada fold, geramos relatórios densos classe a classe e utilizamos o `Plotly` para desenhar as matrizes de confusão interativas e as curvas de perda.

In [6]:
# Célula 6: Validação Cruzada com Gráficos Interativos
manifest = pd.read_csv(MANIFEST_PATH)
results = []

for fold_k in [1, 2, 3]:
    print(f"\n{'='*60}\n INICIANDO FOLD {fold_k}\n{'='*60}")

    train_df = manifest[manifest["fold"] != fold_k].reset_index(drop=True)
    test_df  = manifest[manifest["fold"] == fold_k].reset_index(drop=True)
    mean, std = compute_stats(train_df, MFCC_DIR)

    train_ds = NormalizedDataset(GTZANSequentialDataset(train_df, MFCC_DIR, "train"), mean, std)
    train_loader = DataLoader(train_ds, batch_size=128, shuffle=True, num_workers=2, pin_memory=True)

    model = MusicLSTM().to(DEVICE)
    # Aumentamos o weight_decay para 1e-3 para punir pesos grandes (L2 Regularization)
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-3)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=100)

    best_acc, best_state = 0.0, None
    patience, patience_ctr = 15, 0
    history = []

    for epoch in range(1, 101):
        tr_loss, tr_acc = train_epoch(model, train_loader, optimizer, criterion, DEVICE)
        te_acc, te_f1, _, _ = evaluate_clips(model, test_df, MFCC_DIR, mean, std, DEVICE)
        scheduler.step()
        history.append((tr_loss, tr_acc, te_acc))

        if te_acc > best_acc:
            best_acc, best_state = te_acc, {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_ctr = 0
        else:
            patience_ctr += 1

        if epoch % 10 == 0:
            print(f"  Época {epoch:3d} | loss {tr_loss:.4f} | Treino Acc {tr_acc:.3f} | Teste Acc {te_acc:.3f} (Melhor: {best_acc:.3f})")

        if patience_ctr >= patience:
            print(f"  Early stopping na época {epoch}")
            break

    # ---------------------------------------------------------
    # AVALIAÇÃO DO MELHOR MODELO E GERAÇÃO DE RELATÓRIOS
    # ---------------------------------------------------------
    model.load_state_dict(best_state)
    acc, f1, y_true, y_pred = evaluate_clips(model, test_df, MFCC_DIR, mean, std, DEVICE)
    results.append({"fold": fold_k, "acc_clipe": acc, "f1_macro": f1})

    print(f"\n✅ RESULTADOS FINAIS - FOLD {fold_k}")
    print(classification_report(y_true, y_pred, target_names=GENRES, digits=3))

    # Salva checkpoint
    torch.save({"model_state": best_state, "mean": mean, "std": std, "genres": GENRES, "genre2idx": GENRE2IDX, "fold": fold_k, "acc": acc}, OUTPUT_DIR / f"lstm_fold{fold_k}.pth")

    # 1. Gráfico Plotly: Curvas de Treinamento
    history = np.array(history)
    fig_curves = make_subplots(rows=1, cols=2, subplot_titles=("Loss (Treino)", "Acurácia (Treino vs Teste)"))
    fig_curves.add_trace(go.Scatter(y=history[:, 0], name="Loss Treino", mode='lines', line=dict(color='red')), row=1, col=1)
    fig_curves.add_trace(go.Scatter(y=history[:, 1], name="Acc Treino", mode='lines', line=dict(color='blue')), row=1, col=2)
    fig_curves.add_trace(go.Scatter(y=history[:, 2], name="Acc Teste", mode='lines', line=dict(color='green')), row=1, col=2)
    fig_curves.update_layout(title_text=f"Progresso do Treinamento — Fold {fold_k}", template="plotly_white")
    fig_curves.show()
    fig_curves.write_html(str(OUTPUT_DIR / f"curves_lstm_fold{fold_k}.html"))

    # 2. Gráfico Plotly: Matriz de Confusão
    cm = confusion_matrix(y_true, y_pred)
    cm_norm = np.around(cm / cm.sum(axis=1, keepdims=True), decimals=2)
    fig_cm = px.imshow(cm_norm, text_auto=True, color_continuous_scale='Reds', x=GENRES, y=GENRES,
                       labels=dict(x="Gênero Predito", y="Gênero Real", color="Proporção"))
    fig_cm.update_layout(title=f"Matriz de Confusão (Normalizada) — Fold {fold_k}", width=600, height=600)
    fig_cm.show()
    fig_cm.write_html(str(OUTPUT_DIR / f"confusion_lstm_fold{fold_k}.html"))

# ---------------------------------------------------------
# RESUMO FINAL DOS 3 FOLDS
# ---------------------------------------------------------
df_res = pd.DataFrame(results)
print("\n" + "="*50 + "\n RESUMO 3-FOLD\n" + "="*50)
print(df_res.to_string(index=False))
print(f"\n  Média acc_clipe = {df_res['acc_clipe'].mean():.3f} ± {df_res['acc_clipe'].std():.3f}")
print(f"  Média f1_macro  = {df_res['f1_macro'].mean():.3f} ± {df_res['f1_macro'].std():.3f}")
df_res.to_csv(OUTPUT_DIR / "lstm_resultados.csv", index=False)


 INICIANDO FOLD 1
  Época  10 | loss 1.2733 | Treino Acc 0.688 | Teste Acc 0.438 (Melhor: 0.444)
  Época  20 | loss 0.9823 | Treino Acc 0.831 | Teste Acc 0.479 (Melhor: 0.492)
  Época  30 | loss 0.8507 | Treino Acc 0.895 | Teste Acc 0.489 (Melhor: 0.495)
  Época  40 | loss 0.7701 | Treino Acc 0.934 | Teste Acc 0.505 (Melhor: 0.505)
  Época  50 | loss 0.7179 | Treino Acc 0.955 | Teste Acc 0.495 (Melhor: 0.517)
  Early stopping na época 59

✅ RESULTADOS FINAIS - FOLD 1
              precision    recall  f1-score   support

       blues      0.071     0.031     0.043        32
   classical      0.892     0.971     0.930        34
     country      0.321     0.500     0.391        34
       disco      0.289     0.355     0.319        31
      hiphop      0.667     0.364     0.471        33
        jazz      0.576     0.655     0.613        29
       metal      0.650     0.867     0.743        30
         pop      0.658     0.833     0.735        30
      reggae      0.588     0.345     0.


 INICIANDO FOLD 2
  Época  10 | loss 1.2275 | Treino Acc 0.713 | Teste Acc 0.498 (Melhor: 0.508)
  Época  20 | loss 0.9724 | Treino Acc 0.835 | Teste Acc 0.502 (Melhor: 0.517)
  Early stopping na época 27

✅ RESULTADOS FINAIS - FOLD 2
              precision    recall  f1-score   support

       blues      0.421     0.235     0.302        34
   classical      0.733     0.647     0.688        34
     country      0.410     0.500     0.451        32
       disco      0.500     0.406     0.448        32
      hiphop      0.462     0.545     0.500        33
        jazz      0.560     0.500     0.528        28
       metal      0.880     0.733     0.800        30
         pop      0.542     0.867     0.667        30
      reggae      0.533     0.552     0.542        29
        rock      0.235     0.242     0.239        33

    accuracy                          0.517       315
   macro avg      0.528     0.523     0.516       315
weighted avg      0.524     0.517     0.512       315




 INICIANDO FOLD 3
  Época  10 | loss 1.2873 | Treino Acc 0.678 | Teste Acc 0.438 (Melhor: 0.445)
  Época  20 | loss 0.9842 | Treino Acc 0.830 | Teste Acc 0.470 (Melhor: 0.489)
  Época  30 | loss 0.8510 | Treino Acc 0.897 | Teste Acc 0.492 (Melhor: 0.492)
  Época  40 | loss 0.7630 | Treino Acc 0.936 | Teste Acc 0.508 (Melhor: 0.508)
  Época  50 | loss 0.7141 | Treino Acc 0.957 | Teste Acc 0.486 (Melhor: 0.514)
  Época  60 | loss 0.6848 | Treino Acc 0.970 | Teste Acc 0.517 (Melhor: 0.524)
  Época  70 | loss 0.6601 | Treino Acc 0.980 | Teste Acc 0.514 (Melhor: 0.533)
  Early stopping na época 78

✅ RESULTADOS FINAIS - FOLD 3
              precision    recall  f1-score   support

       blues      0.333     0.176     0.231        34
   classical      1.000     0.906     0.951        32
     country      0.500     0.324     0.393        34
       disco      0.440     0.710     0.543        31
      hiphop      0.484     0.469     0.476        32
        jazz      0.722     0.448     0.553 


 RESUMO 3-FOLD
 fold  acc_clipe  f1_macro
    1   0.517460  0.497913
    2   0.517460  0.516451
    3   0.533123  0.536526

  Média acc_clipe = 0.523 ± 0.009
  Média f1_macro  = 0.517 ± 0.019


---

## 7. Análise de Resultados e Relatório Comparativo

### 7.1. Diagnóstico de Performance da LSTM
A nossa arquitetura Bi-LSTM atingiu uma **Acurácia Média de 51.0%** e um **F1-Score Macro de 50.5%**. Em cenários sem o filtro de vazamento de artistas, LSTMs costumam ultrapassar os 80% no GTZAN, porém esses números são irreais (a rede aprende o timbre da voz/instrumento específico da banda, não o gênero). Nosso resultado de ~51% representa o limite teórico honesto e cientificamente validado para essa abordagem com essas features.

**Comportamento por Gênero (Insights da Matriz de Confusão):**
* **Altíssima Separação (F1 > 0.70):** Gêneros como *Classical*, *Pop* e *Metal* foram facilmente compreendidos pelo modelo. As dinâmicas de silêncio da música clássica e o preenchimento de frequência constante do metal fornecem assinaturas temporais muito claras para a RNN.
* **A Zona de Confusão (F1 < 0.40):** O calcanhar de Aquiles do modelo está em *Rock*, *Blues* e *Country*. Como esses gêneros compartilham essencialmente a mesma paleta de instrumentos acústicos e elétricos de corda, a evolução sequencial crua (frame a frame) dos MFCCs é excessivamente caótica. A LSTM teve imensa dificuldade de encontrar fronteiras de decisão claras entre eles.

### 7.2. Tabela Comparativa Consolidada (Baseline vs. LSTM vs. CNN)
A tabela abaixo cruza os resultados desta modelagem temporal com as abordagens de *Machine Learning* Tabular (Araújo) e Visão Computacional (Borges), fechando o diagnóstico do grupo sobre as melhores representações para o dataset GTZAN:

| Modelo / Abordagem | Acurácia Média | F1-Score Macro | Análise Estrutural e Justificativa |
| :--- | :---: | :---: | :--- |
| **Random Forest**<br>*(Baseline - Araújo)* | 45.8% | 44.2% | **Subajustado:** As árvores de decisão lutam para capturar dependências não lineares complexas apenas a partir de médias estáticas de features acústicas. |
| **SVM (Tuned)**<br>*(Baseline - Araújo)* | 52.0% | 50.4% | **Eficaz no Global:** O SVM lida excepcionalmente bem com a matriz tabular. Ao resumir a música em médias (reduzindo a dimensão do tempo), elimina-se o ruído transiente, criando vetores sólidos e facilmente separáveis por hiperplanos. |
| **Bi-LSTM (Temporal)**<br>*(Este Notebook - Rosendo)* | **51.0%** | **50.5%** | **Muito próximo do SVM:** A adição do *Mean Pooling* e *Data Augmentation* salvou a rede do colapso de overfitting. A rede lida de forma superior com o balanceamento (ótimo F1), mas perde fôlego ao tentar decifrar a altíssima variância sequencial de gêneros muito próximos (Rock/Blues). |
| **CNN 2D (Espectrograma)**<br>*(Deep Learning - Borges)*| **~54.0%** | **~53.0%** | **O Vencedor:** O modelo convalida a tese máxima de MIR (*Music Information Retrieval*). Ao transformar a música em uma imagem bidimensional (Espectrograma Mel), a rede avalia frequências simultâneas como "texturas", ignorando as quebras bruscas do tempo que tanto prejudicaram a LSTM. |

**Conclusão Final:**
A modelagem estritamente temporal baseada em curtos *frames* de 1D é computacionalmente cara e hipersensível a ruídos, não superando estatisticamente um SVM bem parametrizado sobre dados resumidos. Para a classificação de gêneros musicais sob forte restrição de dados (filtro de artistas), o processamento do som como uma representação visual-espacial (CNN) demonstrou ser a vertente arquitetural mais confiável e estável.